# Sesión 4 · Ejercicio 2 — El juicio
**Objetivo:** decidir CON NÚMEROS si sus datos sintéticos de S2-E2
sirven: el protocolo *Train on Synthetic, Test on Real* (TSTR).
**Tiempo:** MÍNIMO 30 min · COMPLETO 60 min
**Produce:** §8 de su bitácora — la tabla de 3 renglones + el cobro +
su conclusión honesta
**Necesitas:** `sinteticos_s2.pt` (el archivo que guardó en S2-E2)


### Cómo trabajar este cuaderno (1 minuto de lectura)

1. **Guarde su copia**: Archivo → Guardar una copia en Drive. Si no, pierde su trabajo al cerrar.
2. Ejecute las celdas **en orden**. Solo las marcadas `#### OBLIGATORIO ####` producen su entregable; las de **EXTENSIÓN** son opcionales, para quien le sobre tiempo.
3. ¿Algo no corre, o tarda demasiado? Ejecute la **CELDA DE RESCATE**: carga resultados ya calculados y usted sigue con el análisis. Usarla **no descuenta puntos** — solo dígalo en su bitácora.
4. Al terminar, copie la figura y sus observaciones (2-3 líneas con sus palabras) a la sección de su **bitácora** que dice el encabezado. Eso es TODO el entregable — no se pide nada más.


In [ ]:
#### OBLIGATORIO #### — setup (idempotente: puede ejecutarla dos veces)
import os, sys
if not os.path.isdir("src"):
    if not os.path.isdir("IAA6_M13_Gen"):
        !git clone -q https://github.com/AdriannaGmz/IAA6_M13_Gen
    %cd IAA6_M13_Gen
!pip install -q -r requirements.txt
sys.path.insert(0, ".")
from src import datos, modelos, evaluar, graficas, rescate
MODO_GPU = rescate.hay_gpu()   # imprime "GPU disponible" o "Modo CPU"


In [ ]:
#### OBLIGATORIO #### — los datos del módulo y sus sintéticos de la Sesión 2
import torch
# La modalidad no se escribe a mano: se lee del propio archivo que
# guardó en S2-E2, para que sus sintéticos y sus reales no puedan
# quedar en modalidades distintas.
try:
    paquete = torch.load("sinteticos_s2.pt", weights_only=False)
    MODO, X_sint = paquete["modalidad"], paquete["X_sint"]
    print(f"Sintéticos propios: {len(X_sint)} de {paquete['clase']!r} "
          f"· modalidad {MODO!r}")
except FileNotFoundError:
    contenido, _ = rescate.cargar("s2_sintetico_generico")
    MODO, X_sint = contenido["modalidad"], contenido["X_sint"]
    print(f"(No encontré su archivo; uso los sintéticos genéricos, "
          f"modalidad {MODO!r}.)")
X, y, meta = datos.cargar(MODO)
idx_min = meta["nombres_clases"].index(meta["clase_minoritaria"])
y_sint = torch.full((len(X_sint),), idx_min)


In [ ]:
#### OBLIGATORIO #### — partición estratificada (jueces aparte)
# La prueba se aparta ANTES de tocar nada: los sintéticos jamás se
# evalúan contra sí mismos.
# COMPLETAR: parta (X, y) en entrenamiento y prueba, 20 % de prueba,
# estratificado (¡la minoritaria debe existir en ambos lados!).
# Pista: la función está en src/datos y devuelve 4 cosas.
X_ent, y_ent, X_pru, y_pru = ...
print(f"entrenamiento: {len(X_ent)} · prueba: {len(X_pru)}")


In [ ]:
#### OBLIGATORIO #### — el protocolo completo: 3 clasificadores idénticos
# linea_base: sólo reales · tstr: sólo sintéticos · aumentado: ambos.
# COMPLETAR: corra el protocolo TSTR con foco en su clase minoritaria.
# Pista: evaluar.tstr(reales_ent, sus_y, sinteticos, sus_y_sint,
#                     prueba, y_prueba, clase_foco=...)
#        Devuelve un dict con "linea_base", "tstr" y "aumentado".
resultados = ...
print(f"LÍNEA BASE (sólo reales) → F1 minoritaria: "
      f"{resultados['linea_base']:.3f}")


In [ ]:
#### OBLIGATORIO #### — TSTR: ¿un clasificador que sólo vio mentiras?
# Entrenado ÚNICAMENTE con sintéticos, evaluado sobre reales.
print(f"TSTR (sólo sintéticos)   → F1 minoritaria: "
      f"{resultados['tstr']:.3f}")
# OJO — lea antes de juzgar el número: sus sintéticos son de UNA sola
# clase, así que este clasificador nunca vio las demás y predice
# "minoritaria" para todo (precisión bajísima). Un TSTR justo pide
# sintéticos de TODAS las clases (Esteban et al. lo hacen así). Con
# una sola clase, el renglón informativo es el AUMENTADO (siguiente).


In [ ]:
#### OBLIGATORIO #### — AUMENTADO: ¿reales + sintéticos ayudan?
print(f"AUMENTADO (reales + sintéticos) → F1 minoritaria: "
      f"{resultados['aumentado']:.3f}")
print("Ésta es LA pregunta del mini-proyecto: ¿generar datos de la")
print("clase escasa mejora al clasificador que la busca?")


In [ ]:
#### OBLIGATORIO #### — la tabla que va a su bitácora
tabla = evaluar.tabla_resultados(resultados)


In [ ]:
#### OBLIGATORIO #### — ★ el cobro: la GAN colapsada de S2-E1 a juicio
# Las GAN de S2 son incondicionales: sus muestras no traen etiqueta.
# Se etiquetan con un clasificador entrenado en reales
# (pseudo-etiquetas) y el protocolo corre igual.
from sklearn.ensemble import RandomForestClassifier
Xi, yi, mi = datos.cargar("imagen")
Xi_ent, yi_ent, Xi_pru, yi_pru = datos.dividir(Xi, yi, prop_prueba=0.2)
etiquetador = RandomForestClassifier(n_estimators=100, random_state=0,
                                     n_jobs=-1)
etiquetador.fit(Xi_ent.flatten(1).numpy(), yi_ent.numpy())

juicio = {}
for nombre in ("s2_gan", "s2_gan_colapso"):
    c, _ = rescate.cargar(nombre)
    g = modelos.GAN(mi, dim_ruido=64)
    g.generador.load_state_dict(c["state_dicts"]["generador"])
    muestras = g.muestrear(300)
    pseudo = torch.tensor(
        etiquetador.predict(muestras.flatten(1).numpy()))
    juicio[nombre] = evaluar.tstr(Xi_ent, yi_ent, muestras, pseudo,
                                  Xi_pru, yi_pru)["tstr"]
print(f"F1 macro TSTR · GAN equilibrada: {juicio['s2_gan']:.3f}")
print(f"F1 macro TSTR · GAN colapsada:   {juicio['s2_gan_colapso']:.3f}")
print("El colapso que MIDIÓ en S2 (cobertura) hoy es F1 desplomado.")


In [ ]:
# ── CELDA DE RESCATE ────────────────────────────────────────
# ¿Algo no corrió? Esto trae todos los números precomputados
# (protocolo tabular + el cobro) y puede escribir su conclusión.
contenido, figuras = rescate.cargar("s4_tstr")
resultados = contenido["resultados"]
juicio = contenido["juicio"]
# `tabla` se asigna aquí también: es lo que imprime la última celda.
tabla = evaluar.tabla_resultados(resultados)
print(f"cobro · GAN equilibrada {juicio['s2_gan']:.3f} · "
      f"colapsada {juicio['s2_gan_colapso']:.3f}")


### Conclusión honesta — el renglón que más pesa de su bitácora

Complete en §8, con la tabla pegada arriba:

- ¿El AUMENTADO superó a la LÍNEA BASE? ¿Por cuánto? ___
- Si su línea base ya era casi perfecta: ¿había algo que mejorar?
  (Con el conjunto de demostración pasa: F1 = 1.0. Dígalo tal cual.) ___
- ¿Recomendaría usar estos sintéticos en producción? ¿Por qué? ___
- ¿Qué relación ve entre la cobertura de modos que midió en S2-E1 y
  el F1 del cobro de hoy? ___

> **Si el resultado salió peor que la línea base, escríbalo. Es un
> resultado válido y se califica igual. Lo que se evalúa es la
> honestidad del análisis.**


In [ ]:
#### OBLIGATORIO #### — artefacto para la bitácora
print("Copie este bloque en la sección §8 de su bitácora:\n")
print(tabla)
print(f"\n- Cobro · GAN equilibrada: {juicio['s2_gan']:.3f} · "
      f"colapsada: {juicio['s2_gan_colapso']:.3f}")
print("- ¿Aumentar ayudó?: <sí/no, por cuánto>")
print("- ¿Lo usaría en producción?: <...>")


### EXTENSIÓN (equipos rápidos)
Varíe la proporción real:sintético del conjunto AUMENTADO (por
ejemplo 1:0.5, 1:1, 1:2 sobre la clase minoritaria) y grafique F1
contra proporción. ¿Más sintéticos siempre ayuda, o hay un punto de
retorno decreciente?
